In [ ]:
!pip install gradio tensorflow matplotlib opencv-python pillow xgboost scikit-learn imbalanced-learn --quiet

In [ ]:
# ==========================================================
# Imports
# ==========================================================
import gradio as gr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile, os, shutil, traceback, cv2
from PIL import Image

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

In [ ]:
# ==========================================================
# Spiral/Wave Detection Module — load a pretrained .keras CNN
# ==========================================================
cnn_model = None
model_input_size = (224, 224)

def load_cnn_model(model_file):
    global cnn_model
    global model_input_size

    try:
        if model_file is None:
            return "Upload a .keras model first."

        model_path = model_file if isinstance(model_file, str) else model_file.name

        cnn_model = tf.keras.models.load_model(model_path, compile=False)

        _, h, w, c = cnn_model.input_shape
        model_input_size = (h, w)

        return f"""
Model Loaded Successfully

Input Size : {model_input_size}

Model :
{cnn_model.name}
"""

    except Exception as e:
        return traceback.format_exc()

In [ ]:
# ==========================================================
# Helpers for spiral/wave image prediction
# ==========================================================
def list_images(folder):
    exts = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
    images = []
    for root, dirs, files in os.walk(folder):
        for f in files:
            if f.lower().endswith(exts):
                images.append(os.path.join(root, f))
    return sorted(images)


def predict_images_show(zip_file):
    global cnn_model
    global model_input_size

    if cnn_model is None:
        return [], "Load CNN model first."

    if zip_file is None:
        return [], "Upload ZIP file."

    extract_dir = "prediction_images"
    shutil.rmtree(extract_dir, ignore_errors=True)
    os.makedirs(extract_dir)

    zip_path = zip_file if isinstance(zip_file, str) else zip_file.name

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)

    image_paths = list_images(extract_dir)

    if len(image_paths) == 0:
        return [], "No images found."

    gallery = []
    healthy = 0
    parkinson = 0

    for path in image_paths:
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize(model_input_size)
            arr = np.array(img).astype(np.float32)
            arr = preprocess_input(arr)
            arr = np.expand_dims(arr, 0)

            pred = cnn_model.predict(arr, verbose=0)

            if pred.shape[-1] == 1:
                probability = float(pred[0][0])
                if probability >= 0.5:
                    label = "Parkinson's"
                    confidence = probability
                    parkinson += 1
                else:
                    label = "Healthy"
                    confidence = 1 - probability
                    healthy += 1
            else:
                idx = np.argmax(pred)
                confidence = float(np.max(pred))
                if idx == 1:
                    label = "Parkinson's"
                    parkinson += 1
                else:
                    label = "Healthy"
                    healthy += 1

            gallery.append((path, f"{label}\nConfidence : {confidence*100:.2f}%"))

        except Exception as e:
            print(e)

    status = f"""
Prediction Completed

Images Processed : {len(gallery)}

Healthy : {healthy}

Parkinson's : {parkinson}
"""

    return gallery, status

In [ ]:
# ==========================================================
# Voice CSV CNN architecture
# Edit here to change filters, dropout rate, or L2 strength
# ==========================================================
def build_cnn(input_len):
    model = models.Sequential([
        layers.Conv1D(32, 3, activation='relu', input_shape=(input_len, 1)),
        layers.BatchNormalization(),
        layers.Conv1D(64, 3, activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.3),                       # increase/decrease if overfitting
        layers.Flatten(),
        layers.Dense(32, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-3)),  # L2 strength here
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
# ==========================================================
# Reorder features by correlation clustering
# Groups related features next to each other, which helps
# a 1D convolution find meaningful local patterns.
# Edit here to change the linkage method.
# ==========================================================
def get_feature_order(X_train_df):
    corr = X_train_df.corr().abs()
    dist = 1 - corr
    np.fill_diagonal(dist.values, 0)
    condensed = squareform(dist.values, checks=False)
    Z = linkage(condensed, method='average')       # try 'ward' or 'complete' too
    order = dendrogram(Z, no_plot=True)['leaves']
    return order

In [ ]:
# ==========================================================
# Load CSV, split, scale, and compute the CNN feature order
# Edit here to change the train/test split ratio
# ==========================================================
def prepare_data(csv_file):
    df = pd.read_csv(csv_file.name)
    if 'name' in df.columns:
        df = df.drop(['name'], axis=1)

    X = df.drop(['status'], axis=1)
    y = df['status']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y   # stratify keeps class balance
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # feature order for the CNN branch only — classical models keep the original order
    feature_order = get_feature_order(X_train)
    X_train_scaled_cnn = X_train_scaled[:, feature_order]
    X_test_scaled_cnn = X_test_scaled[:, feature_order]

    return (X_train_scaled, X_test_scaled,
            X_train_scaled_cnn, X_test_scaled_cnn,
            y_train, y_test)

In [ ]:
# ==========================================================
# Classical ML models (easy to add/remove a model here)
# ==========================================================
def train_classical_models(X_train_scaled, X_test_scaled, y_train, y_test):
    models_dict = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "KNN": KNeighborsClassifier(),
        "SVM": SVC(probability=True),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Gaussian NB": GaussianNB(),
        "XGBoost": XGBClassifier(eval_metric='logloss', use_label_encoder=False),
    }

    results = {}
    for name, model in models_dict.items():
        model.fit(X_train_scaled, y_train)
        acc = model.score(X_test_scaled, y_test)
        results[name] = acc

    return results

In [ ]:
# ==========================================================
# CNN cross-validation — honest baseline estimate
# (single model per fold, no ensembling)
# Edit here to change number of folds or patience
# ==========================================================
def train_cnn_cv(X_train_scaled_cnn, y_train, n_splits=5, patience=10):
    X_train_cnn = np.expand_dims(X_train_scaled_cnn, axis=2)

    classes = np.unique(y_train)
    weights = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight = {int(c): float(w) for c, w in zip(classes, weights)}

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    y_train_arr = y_train.values if hasattr(y_train, "values") else y_train

    cv_accuracies = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_cnn, y_train_arr)):
        print(f"[CV] training fold {fold+1}/{n_splits}...")
        model = build_cnn(X_train_cnn.shape[1])

        sw_train = np.array([class_weight[int(label)] for label in y_train_arr[tr_idx]])

        early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

        model.fit(
            X_train_cnn[tr_idx], y_train_arr[tr_idx],
            validation_data=(X_train_cnn[val_idx], y_train_arr[val_idx]),
            sample_weight=sw_train,
            epochs=100,
            batch_size=16,
            callbacks=[early_stop],
            verbose=0
        )

        _, val_acc = model.evaluate(X_train_cnn[val_idx], y_train_arr[val_idx], verbose=0)
        cv_accuracies.append(val_acc)
        print(f"[CV] fold {fold+1}/{n_splits} done — val accuracy: {val_acc*100:.2f}%")

    print("[CV] finished.\n")
    return float(np.mean(cv_accuracies)), float(np.std(cv_accuracies))

In [ ]:
# ==========================================================
# CNN ensemble (bagging + random feature subspace)
# Each model trains on: a bootstrap sample of rows, a random
# subset of feature columns (like Random Forest), and SMOTE
# balancing. Final prediction is a validation-accuracy-weighted
# average across models — this is what pushes accuracy and
# stability further than plain bagging.
# Edit here to change n_models, feature_frac, or patience
# ==========================================================
def train_cnn_ensemble(X_train_scaled_cnn, y_train, n_models=8, patience=10, feature_frac=0.85):
    X_train_cnn = np.expand_dims(X_train_scaled_cnn, axis=2)
    y_train_arr = y_train.values if hasattr(y_train, "values") else y_train

    n_features = X_train_cnn.shape[1]
    n_selected = max(3, int(round(n_features * feature_frac)))

    trained_models = []
    feature_indices_list = []
    val_accs = []

    for i in range(n_models):
        print(f"[Ensemble] training model {i+1}/{n_models}...")
        rng = np.random.RandomState(i)

        # random feature subspace (Random-Forest-style diversity)
        feat_idx = np.sort(rng.choice(n_features, size=n_selected, replace=False))

        # bootstrap sample of rows
        boot_idx = rng.choice(len(X_train_cnn), size=len(X_train_cnn), replace=True)
        X_boot = X_train_cnn[boot_idx][:, feat_idx, :]
        y_boot = y_train_arr[boot_idx]

        # SMOTE needs 2D input, so flatten/reshape around it
        X_boot_flat = X_boot.reshape(X_boot.shape[0], -1)
        sm = SMOTE(random_state=i)
        X_res_flat, y_res = sm.fit_resample(X_boot_flat, y_boot)
        X_res = X_res_flat.reshape(-1, n_selected, 1)

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_res, y_res, test_size=0.15, random_state=i, stratify=y_res
        )

        model = build_cnn(n_selected)

        early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5)

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=100,
            batch_size=16,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )

        val_probs = model.predict(X_val, verbose=0).flatten()
        val_preds = (val_probs >= 0.5).astype(int)
        val_acc = float(np.mean(val_preds == y_val))

        trained_models.append(model)
        feature_indices_list.append(feat_idx)
        val_accs.append(val_acc)
        print(f"[Ensemble] model {i+1}/{n_models} done — val accuracy: {val_acc*100:.2f}%")

    print("[Ensemble] finished.\n")
    return trained_models, feature_indices_list, val_accs


def ensemble_predict_proba(trained_models, feature_indices_list, val_accs, X):
    # weight each model's vote by how well it did on its own validation split
    weights = np.array(val_accs)
    weights = weights / weights.sum()

    weighted_preds = []
    for model, feat_idx, w in zip(trained_models, feature_indices_list, weights):
        X_sub = X[:, feat_idx, :]
        p = model.predict(X_sub, verbose=0).flatten()
        weighted_preds.append(p * w)

    return np.sum(weighted_preds, axis=0)


def ensemble_evaluate(trained_models, feature_indices_list, val_accs, X_test_scaled_cnn, y_test):
    X_test_cnn = np.expand_dims(X_test_scaled_cnn, axis=2)
    y_test_arr = y_test.values if hasattr(y_test, "values") else y_test

    probs = ensemble_predict_proba(trained_models, feature_indices_list, val_accs, X_test_cnn)
    preds = (probs >= 0.5).astype(int)
    acc = float(np.mean(preds == y_test_arr))
    return acc

In [ ]:
# ==========================================================
# Main entry point — wired to the Gradio button
# ==========================================================
def train_and_evaluate(csv_file):
    try:
        (X_train_scaled, X_test_scaled,
         X_train_scaled_cnn, X_test_scaled_cnn,
         y_train, y_test) = prepare_data(csv_file)

        print("Training classical models...")
        results = train_classical_models(X_train_scaled, X_test_scaled, y_train, y_test)
        print("Classical models done.\n")

        cnn_cv_mean, cnn_cv_std = train_cnn_cv(X_train_scaled_cnn, y_train)

        cnn_models, cnn_feature_idx, cnn_val_accs = train_cnn_ensemble(X_train_scaled_cnn, y_train)
        cnn_ensemble_acc = ensemble_evaluate(cnn_models, cnn_feature_idx, cnn_val_accs, X_test_scaled_cnn, y_test)

        results["CNN (Ensemble)"] = cnn_ensemble_acc

        report_lines = ["Final Accuracy Report:"]
        for model_name, acc in results.items():
            report_lines.append(f"Accuracy Score of {model_name}: {acc*100:.2f}%")
        report_lines.append(
            f"\nCNN cross-validation accuracy (single model, honest estimate): "
            f"{cnn_cv_mean*100:.2f}% +/- {cnn_cv_std*100:.2f}%"
        )
        report_text = "\n".join(report_lines)

        fig, ax = plt.subplots(figsize=(6, 4))
        ax.barh(list(results.keys()), list(results.values()))
        ax.set_xlabel("Accuracy")
        ax.set_title("Model Comparison")
        plt.tight_layout()

        return report_text, fig

    except Exception as e:
        return f"Error: {str(e)}", None

In [ ]:
# ==========================================================
# Custom CSS for the Gradio UI
# ==========================================================
custom_css = """
body, .gradio-container {
    background: linear-gradient(135deg, #0f2027, #203a43, #2c5364) !important;
    color: white !important;
}
h1, h2, h3, p, .gradio-container > label {
    color: white !important;
}
.gr-button {
    background-color: #29b6f6 !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: bold !important;
    color: white !important;
}
.gr-button:hover {
    background-color: #0288d1 !important;
}
.gr-tab {
    font-size: 1.2rem !important;
    padding: 15px !important;
}
/* Fix invisible gallery caption text under spiral images */
.caption-label, .thumbnail-caption, .gallery .caption, .gallery-item .caption {
    color: #000 !important;
    background: rgba(255,255,255,0.9) !important;
    font-weight: 600 !important;
}
"""

In [ ]:
# ==========================================================
# Gradio UI (tabs below heading, no separate results tab)
# ==========================================================
with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:
    # === HEADER ===
    gr.HTML("""
    <div style='text-align:center; padding:25px; background:rgba(255,255,255,0.1); border-radius:15px;'>
        <h1 style='font-size:3em;'>Parkinson's Disease Detection System</h1>
        <p style='font-size:1.3em;'>
            A unified platform using <b>Machine Learning</b> and <b>Deep Learning</b>
            to analyze <b>voice</b> and <b>hand-drawn spirals/waves</b> for early detection of Parkinson's disease.
        </p>
        <p style='font-size:1em; opacity:0.9;'>
            Developed for research and healthcare analysis — fast, interpretable, and dataset-driven.
        </p>
    </div>
    """)

    # === NAVIGATION TABS ===
    with gr.Tab("Home"):
        gr.Markdown("### Welcome to the Parkinson's Detection Dashboard")

    with gr.Tab("Voice Dataset (CSV)"):
        csv_input = gr.File(label="Upload Voice Dataset (.csv)", file_types=[".csv"])
        submit_btn_csv = gr.Button("Run / Submit")
        output_text = gr.Textbox(label="Final Accuracy Report", lines=14)
        output_plot = gr.Plot(label="Accuracy Comparison")
        submit_btn_csv.click(train_and_evaluate, inputs=[csv_input], outputs=[output_text, output_plot])

    with gr.Tab("Spiral/Wave Images"):
        h5_file = gr.File(label="Upload CNN Model (.keras)", file_types=[".keras"])
        load_btn = gr.Button("Load CNN Model")
        model_status = gr.Textbox(label="Model Status", interactive=False)
        load_btn.click(load_cnn_model, inputs=h5_file, outputs=model_status)

        zip_file = gr.File(label="Upload ZIP of Images", file_types=[".zip"])
        submit_btn_img = gr.Button("Run / Submit")
        gallery = gr.Gallery(label="Results", columns=4)
        submit_btn_img.click(predict_images_show, inputs=[zip_file], outputs=[gallery, model_status])

    # === FOOTER ===
    gr.HTML("<footer style='text-align:center; margin-top:40px; font-size:1rem; opacity:0.8;'>Parkinson's Research UI | Built with Gradio & TensorFlow</footer>")

demo.launch(share=True)